In [ ]:
import pandas as pd
import plotly.express as px

insar = pd.read_csv("../model/insar_features.csv")
reliable = insar[insar["insar_reliable"] == True].copy()

ds1 = pd.read_csv("DS1_all.csv")
ds1["completion_date"] = pd.to_datetime(ds1["NSEARCH3_E"], errors="coerce")
ds1["age"] = (pd.Timestamp.today() - ds1["completion_date"]).dt.days / 365.25

In [ ]:
import plotly.io as pio
import plotly.graph_objects as go

# ── App colour palette ────────────────────────────────────────────────────────
BG       = "#151617"          # presentation background
BG_PLOT  = "#0c111c"          # slightly lighter for plot area (app --bg-2)
TEXT0    = "#e6ecf5"
TEXT1    = "#aab4c4"
TEXT2    = "#6c7689"
ACCENT   = "#6ec1ff"          # app --accent  (replaces steelblue)
CRITICAL = "#E84545"          # app --critical (replaces tomato/red)
HIGH     = "#F37735"          # app --high
MODERATE = "#F5B642"          # app --moderate
LOW      = "#3FB6B0"          # app --low
BORDER   = "rgba(120,160,220,0.13)"
GRIDLINE = "rgba(140,200,255,0.07)"

_t = go.layout.Template()
_t.layout = go.Layout(
    paper_bgcolor=BG,
    plot_bgcolor=BG_PLOT,
    font=dict(color=TEXT0, family="ui-monospace, 'JetBrains Mono', 'SF Mono', Menlo, Consolas, monospace"),
    title=dict(font=dict(color=TEXT0, size=15)),
    xaxis=dict(gridcolor=GRIDLINE, linecolor=BORDER, zerolinecolor=GRIDLINE,
               tickfont=dict(color=TEXT1), title=dict(font=dict(color=TEXT1))),
    yaxis=dict(gridcolor=GRIDLINE, linecolor=BORDER, zerolinecolor=GRIDLINE,
               tickfont=dict(color=TEXT1), title=dict(font=dict(color=TEXT1))),
    legend=dict(bgcolor="rgba(21,22,23,0.85)", bordercolor=BORDER,
                font=dict(color=TEXT1)),
    colorway=[ACCENT, CRITICAL, HIGH, MODERATE, LOW, TEXT2],
    autosize=False, width=1200, height=400,
    colorscale=dict(sequential=[[0, LOW], [0.5, HIGH], [1, CRITICAL]]),
)
pio.templates["app_dark"] = _t
pio.templates.default = "plotly+app_dark"
# Auto-export every figure to plots/ on show (overwrites previous run)
import os as _os, re as _re
import plotly.basedatatypes as _base
_os.makedirs("plots", exist_ok=True)
_plot_n = [0]
_orig_show = _base.BaseFigure.show
def _fixed_show(self, *args, **kwargs):
    _plot_n[0] += 1
    raw = (self.layout.title.text or f"plot_{_plot_n[0]}").strip()
    slug = _re.sub(r"[^\w]+", "_", raw).strip("_")
    self.write_image(f"plots/{slug}.png")
    self.write_image(f"plots/{slug}.svg")
    return _orig_show(self, *args, **kwargs)
_base.BaseFigure.show = _fixed_show

In [ ]:
## Distribution of InSAR velocities (reliable points only)
fig = px.histogram(
    reliable,
    x="insar_vertical_mm_yr",
    nbins=80,
    labels={"insar_vertical_mm_yr": "Vertical velocity (mm/yr)"},
    title="Distribution of vertical InSAR velocities (reliable points)",
)
fig.update_layout(bargap=0.05)
fig.show()

In [ ]:
## Velocity vs. building age
merged = reliable.merge(ds1[["OBJECTID", "age"]], on="OBJECTID", how="inner").dropna(subset=["age"])

fig = px.scatter(
    merged,
    x="age",
    y="insar_vertical_mm_yr",
    opacity=0.4,
    trendline="ols",
    labels={
        "age": "Building age (years)",
        "insar_vertical_mm_yr": "Vertical velocity (mm/yr)",
    },
    title="InSAR vertical velocity vs. building age",
)
fig.show()